# 2 Baseline 冒烟跑通

本章使用 cann-recipes-infer 官方推理框架，在 NPU 上运行 LongCat-Next 的**小说续写请求**（第 3 章同一请求的 3-token 短版本，输出上限由 YAML 控制），验证"权重加载 → prefill → decode → 结果落盘"整条链路。完成后会得到该请求的 `text.txt` 与 `result.json`，作为后续章节的基线。

上游 cann-recipes-infer 仓内置的是 LongCat-Next **多卡**推理实现；本章第 1 节会把该仓**克隆到运行目录并打上课程提供的单卡推理 patch**（`patches/longcat-next-single-npu.patch`，即第 1 章介绍的单卡 CPU offload 实现），后续章节复用同一克隆。


## 1 环境与工作区准备

定位教程目录，创建本次实验的运行目录，并把 CANN 环境变量导入当前 Notebook 进程。本单元还会完成两件一次性准备：① 克隆上游 cann-recipes-infer 仓（全量约 300MB，固定基线提交 `969160e`）并 `git am` 单卡推理 patch；② 检查 1-6 章统一依赖（仅当包缺失或版本不匹配时才安装或替换，全部满足时跳过安装，后续章节直接复用该环境）。运行目录指 `Sources/model_inference_optimization/longcat_next`，用于保存推理仓克隆（`cann-recipes-infer/`）、Notebook 生成的运行 YAML、指标与 Profiling 摘要；课程模板和源码在 `tutorials/llm_inference/longcat_next`。

**推理框架的获取是幂等的**：克隆已就绪则直接复用（不联网、不重复打 patch）；若克隆状态异常（如 patch 应用中断、误切分支）会自动回退基线重新应用，且不影响运行 YAML 与 `res/` 结果等运行产物。

模型权重通过 `LONGCAT_NEXT_MODEL_PATH` 环境变量指定（已有本地权重时直接使用）；**未设置时，本单元会自动从 ModelScope 下载 `meituan-longcat/LongCat-Next`（约 151GB，视带宽约 25 分钟~4 小时，支持断点续传）**——首次运行请确保下载目录所在磁盘有 ≥160GB 可用空间（可用 `LONGCAT_NEXT_DOWNLOAD_DIR` 指定下载目录）。解析成功后路径会缓存在运行目录（`resolved_model_path.txt`），第 3-6 章及后续会话自动复用，不会重复下载。多模态演示素材通过软链 `requests/assets` 指向权重目录自带的官方 `assets/`（book.png、math1.wav 等）。


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

print('[环境准备] 开始', flush=True)
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def locate_repo_root():
    repo_roots = []
    for env_name in ('GITCODE_REPO_ROOT', 'CANN_LEARNING_HUB_ROOT', 'REPO_ROOT'):
        if os.environ.get(env_name):
            repo_roots.append(Path(os.environ[env_name]))
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/longcat_next/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root. Open this notebook inside cann-learning-hub or set CANN_LEARNING_HUB_ROOT to the repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'longcat_next'
SRC_DIR = TUTORIAL_DIR / 'src'
SOURCE_DIR = REPO_ROOT / 'Sources' / 'model_inference_optimization' / 'longcat_next'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment before running this notebook.')
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' not in line:
        continue
    key, value = line.split('=', 1)
    if key == 'PATH':
        # login shell 会触发 conda init 并激活 base，其 PATH 会让后续 %%bash 单元里的
        # python3 解析到 base 环境（本教程环境没有 diffusers 等依赖）。
        # 跳过 PATH，保住当前 kernel 解释器的优先级。
        continue
    os.environ.__setitem__(key, value)
# 把 CANN bin 追加到当前 kernel 的 PATH（不覆盖），infer.sh 内的 python3 仍解析到本环境
_kernel_bin = str(Path(sys.executable).parent)
_cann_bin = str(Path(os.environ.get('ASCEND_HOME_PATH', '/usr/local/Ascend/ascend-toolkit')) / 'bin')
os.environ['PATH'] = f'{_kernel_bin}:{_cann_bin}:{os.environ["PATH"]}'

REQUIREMENTS_FILE = TUTORIAL_DIR / 'requirements.txt'
from inference_scripts.environment_check import find_unmet_requirements, read_pinned_requirements

pinned_requirements = read_pinned_requirements(REQUIREMENTS_FILE)
unmet_requirements = find_unmet_requirements(pinned_requirements)
if unmet_requirements:
    print('[依赖检查] 以下依赖缺失或版本不匹配：')
    for name, actual, expected in unmet_requirements:
        print(f'  - {name}: 当前={actual or "未安装"}, 要求={expected}')
    requirements_to_install = [f'{name}=={expected}' for name, _, expected in unmet_requirements]
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install',
            '-i', 'https://pypi.tuna.tsinghua.edu.cn/simple',
            '--extra-index-url', 'https://download.pytorch.org/whl/cpu',
            '--trusted-host', 'pypi.tuna.tsinghua.edu.cn',
            *requirements_to_install,
        ],
        check=True,
    )
    remaining = find_unmet_requirements(pinned_requirements)
    if remaining:
        details = ', '.join(f'{name}: {actual or "未安装"} != {expected}' for name, actual, expected in remaining)
        raise RuntimeError(f'依赖安装后仍不满足统一版本：{details}')
    print('[依赖检查] 安装完成，统一环境已满足')
else:
    print('[依赖检查] 所有依赖版本均满足，跳过安装')

from inference_scripts.recipe_workflow import (
    ensure_infer_repo,
    ensure_request_assets,
    prepare_runtime_yaml,
    print_yaml_focus,
    resolve_model_path,
)

# 克隆上游 cann-recipes-infer（固定基线 969160e）并应用单卡推理 patch；幂等，已就绪则直接复用
RECIPE_ROOT = ensure_infer_repo()
RECIPE_CONFIG_DIR = RECIPE_ROOT / 'models' / 'longcat_next' / 'config'
RECIPE_REQUESTS_DIR = RECIPE_ROOT / 'models' / 'longcat_next' / 'requests'
INFER_SH = RECIPE_ROOT / 'executor' / 'scripts' / 'infer.sh'
MODEL_PATH = resolve_model_path()
ensure_request_assets(MODEL_PATH, RECIPE_REQUESTS_DIR)

os.environ['LONGCAT_RECIPE_ROOT'] = str(RECIPE_ROOT)
os.environ['LONGCAT_MODEL_PATH'] = MODEL_PATH

print('仓库目录 =', REPO_ROOT)
print('教程目录 =', TUTORIAL_DIR)
print('运行目录 =', SOURCE_DIR)
print('infer 仓目录 =', RECIPE_ROOT)
print('YAML 目录 =', RECIPE_CONFIG_DIR)
print('推理入口 =', INFER_SH)
print('模型权重 =', MODEL_PATH)
print('可见 NPU =', os.environ.get('ASCEND_RT_VISIBLE_DEVICES'))
print('[环境准备] 完成', flush=True)



## 2 查看 YAML 并执行冒烟推理

本节按 recipes 的常规流程执行：先查看并准备 YAML，再通过 `infer.sh` 启动离线推理，最后读取运行结果并整理为课程指标。


### 2.1 准备运行 YAML

冒烟 YAML 位于 `models/longcat_next/config/longcat_next_multimodal_single_npu_offload_smoke.yaml`。手工运行时重点确认以下字段：

| 字段 | 取值 | 说明 |
| --- | --- | --- |
| `model_config.model_path` | 本地权重目录 | 模板中为占位值 `/path/to/LongCat-Next`，Notebook 会自动写入解析结果；手工运行时需改为本地权重目录（15 个 safetensors 分片所在） |
| `model_config.exe_mode` | `eager` | 执行模式，冒烟用 eager |
| `custom_params.enable_moe_cpu_offload` | `True` | MoE 专家权重驻留 CPU，单卡必须开启 |
| `custom_params.enable_ngram_cpu_offload` | `True` | N-gram 嵌入驻留 CPU，单卡必须开启 |
| `custom_params.multimodal.request_file` | `requests/text.json` | 请求清单（与第 3 章共用小说续写请求），相对 `models/longcat_next/` 解析 |
| `data_config.dataset` | `longcat_multimodal` | 多模态请求清单入口，保持不变 |
| `data_config.input_truncated_len` | `64` | 输入容量基数（KV 预留的一部分） |
| `scheduler_config.max_new_tokens` | `3` | 输出上限（同时影响 KV 预留）。请求清单本身不带该字段时由此处控制：本冒烟 YAML 截断为 3，第 3 章 YAML 放开为 128 |

下面的 Python 单元基于模板生成运行 YAML：写入当前模型路径后，保存到**克隆仓的 `models/longcat_next/config/` 目录**（上游 `infer.sh` 的 `--yaml` 参数按仓库约定只接受 `config/` 下的文件名），并打印关键字段。


In [ ]:
import os
import time
from inference_scripts.recipe_workflow import (
    collect_request_metrics,
    find_recipe_case_dir,
    prepare_runtime_yaml,
    print_yaml_focus,
)

BASELINE_WORK_DIR = SOURCE_DIR / 'run_outputs' / 'baseline_smoke'
# 运行 YAML 写入克隆仓 config/ 目录：上游 infer.sh 的 --yaml 只接受 config/ 下的文件名
BASELINE_YAML = RECIPE_CONFIG_DIR / 'baseline_smoke.yaml'
BASELINE_YAML_TEMPLATE = RECIPE_CONFIG_DIR / 'longcat_next_multimodal_single_npu_offload_smoke.yaml'
BASELINE_WORK_DIR.mkdir(parents=True, exist_ok=True)

print('[冒烟推理] 开始', flush=True)
prepare_runtime_yaml(BASELINE_YAML_TEMPLATE, BASELINE_YAML, model_path=MODEL_PATH)
print_yaml_focus(BASELINE_YAML)

os.environ['LONGCAT_RUN_YAML'] = str(BASELINE_YAML)
os.environ['LONGCAT_RUN_YAML_NAME'] = BASELINE_YAML.name  # infer.sh --yaml 传文件名
os.environ['LONGCAT_RUN_LOG'] = str(BASELINE_WORK_DIR / 'infer_stdout.log')
os.environ['LONGCAT_RUN_START_TIME'] = str(time.perf_counter())
print('参考 YAML =', BASELINE_YAML_TEMPLATE)
print('运行 YAML =', BASELINE_YAML)


### 2.2 使用 recipes 命令启动推理

这部分展示 recipes 的运行方式：先进入运行目录中的 cann-recipes-infer 克隆（布局与上游仓一致：executor / module / dataset / models），再通过 `executor/scripts/infer.sh` 启动离线推理。`--yaml` 按上游约定传 `config/` 下的 YAML **文件名**（运行 YAML 已由上一节写入该目录）。代码中会显式打印并执行对应命令。

首次运行包含约 2-3 分钟的权重加载（151GB mmap）；3-token 冒烟生成本身约 30 秒。`infer.sh` 会把日志写入 `models/longcat_next/res/<日期>/<case>/log_0.log`，终端同步 tee 一份。


In [ ]:
%%bash
set -euo pipefail

echo 'recipes 启动命令:'
echo "cd ${LONGCAT_RECIPE_ROOT}"
echo "bash executor/scripts/infer.sh --model longcat_next --yaml ${LONGCAT_RUN_YAML_NAME}"
cd "${LONGCAT_RECIPE_ROOT}"
bash executor/scripts/infer.sh --model longcat_next --yaml "${LONGCAT_RUN_YAML_NAME}" 2>&1 | tee "${LONGCAT_RUN_LOG}"


### 2.3 读取运行结果

推理结束后，recipes 会在 `models/longcat_next/res/<日期>/longcat_next_<yaml名>/` 下生成日志与结果。LongCat-Next 样例为**每条请求**输出结构化的 `result.json`（含生成 token 数、自回归耗时、采样参数等），无需解析日志文本。下面的单元校验日志无错误，并把各请求指标整理为课程 metrics 文件。


In [ ]:
case_dir = find_recipe_case_dir(RECIPE_ROOT, BASELINE_YAML)
if case_dir is None:
    raise RuntimeError(f'未找到 recipes 结果目录：{BASELINE_YAML.stem}')
run_info = {
    'stdout_path': os.environ['LONGCAT_RUN_LOG'],
    'wall_time_s': time.perf_counter() - float(os.environ['LONGCAT_RUN_START_TIME']),
    'recipe_case_dir': str(case_dir),
}
baseline_metrics = collect_request_metrics(
    RECIPE_ROOT,
    BASELINE_WORK_DIR,
    'baseline_smoke',
    BASELINE_YAML,
)
print('请求指标 =', json.dumps(baseline_metrics['requests'], ensure_ascii=False, indent=2))


## 3 查看结果

运行结束后查看请求的生成 token 数与耗时，并与参考基线（Atlas A3 单卡、BF16、eager、CPU offload 开启）对比。首 token 时间即 **TTFT**（prefill 主导）；其余 2 个 token 的平均间隔可粗略观察 **TPOT**（CPU offload 主导）。环境不同时数值会有浮动，量级一致即可。


In [ ]:
print('[结果查看] 开始', flush=True)
print(f"{'请求':<18} {'tokens':>8} {'耗时(s)':>10} {'备注'}")
for row in baseline_metrics['requests']:
    print(f"{row['id']:<18} {row['generated_sequence_tokens']:>8} {row['autoregressive_seconds']:>10.2f} autoregressive")

reference = {
    'text_story': {'tokens': 3, 'seconds': 31.4},
}
print('\n参考基线（Atlas A3 / CANN 8.5.1 / 同配置）:')
for key, value in reference.items():
    print(f"  {key}: {value['tokens']} tok, {value['seconds']} s")

results_dir = Path(baseline_metrics['recipe_case_dir']) / 'results'
for request_dir in sorted(results_dir.iterdir()):
    text_file = request_dir / 'text.txt'
    if text_file.is_file():
        print(f"\n{request_dir.name} 生成文本:")
        print(text_file.read_text(encoding='utf-8'))
print('[结果查看] 完成', flush=True)
